# 03 — Clean invalid records and flag suspicious prices

**Execution status:** saved preparation outputs belong to the local run/configuration used to produce them. The runner saves new executed copies separately.

 **Purpose:** remove invalid prices, exclude Borong/low-coverage items and preserve suspicious prices for review.
 **Saved:** cleaned daily Parquet, cleaning metadata/learned bounds and source/coverage review CSVs.
 A price flag is evidence to inspect, not proof of an error. The default retains flagged prices.

Up to six fixed reviewed records are conservatively excluded in the holdout training period. Expanding folds retain them by default because review dates were not recorded. Other statistical flags are retained. These suspected errors are not externally confirmed; their exclusion policy is documented explicitly. No manual review CSV is required.


**Manual workflow:** run the previous notebook first, then Run All here. This notebook continues the run started by 01; no folder settings are needed. Start again from 01 for a new reproduction.

In [1]:
import pandas as pd
import numpy as np
import json
import hashlib
import pricecatcher_config as cfg
from importlib import reload
cfg = reload(cfg)
cfg.protect_frozen_preparation()
print("Run folder:", cfg.PROJECT_ROOT)
print("Experiment:", cfg.EXPERIMENT, "| Training cutoff:", cfg.TRAIN_END_DATE)

manifest = cfg.verify_source_snapshot()
preparation = json.loads((cfg.STAGE2_DIR / "preparation_settings.json").read_text())
expected = {**cfg.PREPARATION_SETTINGS, "source_manifest_sha256": hashlib.sha256(
    (cfg.STAGE1_DIR / "source_manifest.json").read_bytes()).hexdigest()}
expected["source_snapshot_sha256"] = manifest["source_snapshot_sha256"]
if preparation != expected:
    raise ValueError("Preparation settings/source changed. Rerun 02–05 for this experiment.")
df = pd.read_parquet(cfg.STAGE1_COMBINED_PARQUET)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
if df["date"].isna().any():
    raise ValueError("Missing source dates: rerun/check 01.")
df = df.loc[df["date"].between(pd.Timestamp(cfg.START_MONTH + "-01"),
                                  pd.Timestamp(cfg.EVALUATION_END_DATE))].copy()
original_rows = len(df)
required = ["date", "premise_code"] + cfg.GROUP_KEYS
if df[required].isna().any().any():
    raise ValueError("Missing dates or grouping details: inspect the integration output.")
if df.duplicated(["date", "premise_code", "item_code"]).any():
    raise ValueError("Duplicate price keys need review before cleaning.")

# Small source-level tables keep fully excluded places/series visible in later audits.
geography = ["state", "district"]
daily_geography = df.groupby(geography).size().rename("source_records").to_frame()
source_series = df.groupby(cfg.GROUP_KEYS).agg(
    source_records=("price", "size"), source_observed_days=("date", "nunique"))
source_series.to_csv(cfg.STAGE3_DIR / "source_series_coverage.csv")


Run folder: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240
Experiment: holdout | Training cutoff: 2025-12-31


## 1. Remove invalid prices and apply item/premise exclusions


In [2]:
invalid = df["price"].isna() | ~np.isfinite(df["price"]) | (df["price"] <= 0)
invalid_count = int(invalid.sum())
daily_geography["invalid_removed"] = df.loc[invalid].groupby(geography).size()
invalid_review = df.loc[invalid & (df["date"] <= pd.Timestamp(cfg.TRAIN_END_DATE))].copy()
invalid_review["review_reason"] = "invalid price"
df = df.loc[~invalid].copy()
print("Invalid prices removed:", invalid_count)

coverage = pd.read_csv(cfg.STAGE2_ITEM_COVERAGE_CSV)
if not coverage["training_end"].astype(str).eq(cfg.TRAIN_END_DATE).all():
    raise ValueError("Coverage report uses a different training cutoff. Rerun Notebook 02.")
low_codes = coverage.loc[coverage["exclude_low_coverage"].eq(True), "item_code"].tolist()
borong = df["premise_type"].eq("Borong") if cfg.REMOVE_BORONG else pd.Series(False, index=df.index)
low_coverage = df["item_code"].isin(low_codes) & ~borong
borong_count, low_count = int(borong.sum()), int(low_coverage.sum())
daily_geography["borong_removed"] = df.loc[borong].groupby(geography).size()
daily_geography["low_coverage_item_removed"] = df.loc[low_coverage].groupby(geography).size()
model = df.loc[~borong & ~low_coverage].copy()
if model.empty:
    raise ValueError("No records remain after item/premise exclusions.")
print("Borong excluded:", borong_count, "| Low-coverage records excluded:", low_count)


Invalid prices removed: 0
Borong excluded: 1016 | Low-coverage records excluded: 8295


## 2. Learn price bounds from training retail data
 Bounds use the valid training basket before low-coverage exclusions, preserving the earlier flagging rule. No validation/test prices determine these bounds.


In [3]:
reference = df[df["date"] <= pd.Timestamp(cfg.TRAIN_END_DATE)].copy()
if cfg.REMOVE_BORONG:
    reference = reference[reference["premise_type"] != "Borong"]
if reference.empty:
    raise ValueError("No training data available for price bounds.")

stats = reference.groupby("item_code")["price"].agg(
    q1=lambda x: x.quantile(0.25), median="median", q3=lambda x: x.quantile(0.75))
stats["iqr"] = stats["q3"] - stats["q1"]
stats["lower_bound"] = pd.concat([stats["q1"] - cfg.IQR_MULTIPLIER * stats["iqr"],
                                  stats["median"] * cfg.LOW_MEDIAN_MULTIPLIER], axis=1).max(axis=1).clip(lower=0)
stats["upper_bound"] = pd.concat([stats["q3"] + cfg.IQR_MULTIPLIER * stats["iqr"],
                                  stats["median"] * cfg.HIGH_MEDIAN_MULTIPLIER], axis=1).min(axis=1)
zero_iqr = stats["iqr"] == 0
stats.loc[zero_iqr, "lower_bound"] = stats.loc[zero_iqr, "median"] * cfg.LOW_MEDIAN_MULTIPLIER
stats.loc[zero_iqr, "upper_bound"] = stats.loc[zero_iqr, "median"] * cfg.HIGH_MEDIAN_MULTIPLIER

model = model.merge(stats[["lower_bound", "upper_bound"]], on="item_code", how="left", validate="many_to_one")
model["is_suspicious_price"] = (model["price"] < model["lower_bound"]) | (model["price"] > model["upper_bound"])
print("Flagged modelling records:", int(model["is_suspicious_price"].sum()))
print("Records without training bounds:", int(model["lower_bound"].isna().sum()))


Flagged modelling records: 1933
Records without training bounds: 0


## 2a. Apply training-dated documented conservative exclusions
TIMUN: four RM55 records at premise 19117 follow RM5.50 observations in the same calendar-week pair. Sugar: two RM28.50 records replace RM2.85 while other matched premises remain at RM2.85. Exact tenfold patterns support conservative exclusion, not an invented correction. Bounds above remain fixed so the sensitivity comparison isolates the six-record decision. Other flagged values, including extreme high values, remain unverified and retained; report this residual uncertainty.

Exception discovery was retrospective; fold date checks prevent evaluation-dated exclusions but do not prove real-time error detection.


In [4]:
reviewed_records = pd.DataFrame([
    ["2025-02-27", 19117, 113, 55.0],
    ["2025-02-28", 19117, 113, 55.0],
    ["2025-03-01", 19117, 113, 55.0],
    ["2025-03-02", 19117, 113, 55.0],
    ["2025-09-29", 1291, 1589, 28.5],
    ["2025-09-29", 20872, 1589, 28.5],
], columns=["date", "premise_code", "item_code", "expected_price"])
reviewed_records["date"] = pd.to_datetime(reviewed_records["date"])
reviewed_records["reason"] = "Suspected tenfold source-price issue; conservative exclusion, not confirmed error"
# Do not use an evaluation-period observation to decide a specific source exception.
# These fixed retrospective exceptions remain a research limitation, not confirmed errors.
reviewed_records = reviewed_records.loc[
    reviewed_records["date"].le(pd.Timestamp(cfg.TRAIN_END_DATE))
    & ~reviewed_records["item_code"].isin(low_codes)].copy()
if not cfg.APPLY_DOCUMENTED_EXCLUSIONS:
    reviewed_records = reviewed_records.iloc[0:0].copy()
keys = ["date", "premise_code", "item_code"]
matched = reviewed_records.merge(model, on=keys, how="left", validate="one_to_one")
if matched["price"].isna().any() or not np.isclose(matched["price"], matched["expected_price"]).all():
    raise ValueError("Reviewed records differ from this source snapshot. Check before excluding.")
reviewed_daily = matched[cfg.FINAL_COLS + ["is_suspicious_price"]].copy()
daily_geography["documented_removed"] = reviewed_daily.groupby(geography).size()
model = model.merge(reviewed_records[keys].assign(exclude=True), on=keys, how="left", validate="one_to_one")
model = model.loc[model["exclude"].ne(True)].drop(columns="exclude")
display(reviewed_records)
print("Documented records excluded:", len(reviewed_records))


,date,premise_code,item_code,expected_price,reason
0,2025-02-27,19117,113,55.0,Suspected tenfold source-price issue; conserva...
1,2025-02-28,19117,113,55.0,Suspected tenfold source-price issue; conserva...
2,2025-03-01,19117,113,55.0,Suspected tenfold source-price issue; conserva...
3,2025-03-02,19117,113,55.0,Suspected tenfold source-price issue; conserva...
4,2025-09-29,1291,1589,28.5,Suspected tenfold source-price issue; conserva...
5,2025-09-29,20872,1589,28.5,Suspected tenfold source-price issue; conserva...


Documented records excluded: 6


In [5]:
reviewed_records.to_csv(cfg.STAGE3_DIR / "documented_exclusions.csv", index=False)


## 3. Save daily data and review evidence
Statistical flags are retained by default. The source-review CSV is an evidence export, not a file you must edit. Keep the original integrated dataset.


In [6]:
flagged_review = model.loc[model["is_suspicious_price"] & (model["date"] <= pd.Timestamp(cfg.TRAIN_END_DATE))].copy()
flagged_review["review_reason"] = "outside training price bounds; cause unconfirmed"
review = pd.concat([invalid_review, flagged_review], ignore_index=True)
review.to_csv(cfg.STAGE3_DIR / "training_price_review.csv", index=False)
print("Training records for source review:", len(review))
display(review[cfg.FINAL_COLS + ["review_reason"]].head(10))

flagged_count = int(model["is_suspicious_price"].sum())
flagged_removed = flagged_count if cfg.REMOVE_FLAGGED_PRICES else 0
daily_geography["statistical_flag_removed"] = (
    model.loc[model["is_suspicious_price"]].groupby(geography).size()
    if cfg.REMOVE_FLAGGED_PRICES else 0)
if cfg.REMOVE_FLAGGED_PRICES:
    model = model.loc[~model["is_suspicious_price"]].copy()
model = model[cfg.FINAL_COLS + ["is_suspicious_price"]]
if model.empty:
    raise ValueError("No records remain after price screening.")
model.to_parquet(cfg.STAGE3_MODEL_READY_PARQUET, index=False)
daily_geography["retained_records"] = model.groupby(geography).size()
daily_geography = daily_geography.fillna(0).astype(int)
assert daily_geography.drop(columns="source_records").sum(axis=1).eq(daily_geography["source_records"]).all()
daily_geography.to_csv(cfg.STAGE3_DIR / "daily_geographical_coverage.csv")
summary = {"preparation": preparation,
    "reviewed_records_excluded": len(reviewed_records),
    "target_status": cfg.TARGET_STATUS,
    "reviewed_daily_records": reviewed_daily.assign(date=reviewed_daily["date"].dt.strftime("%Y-%m-%d")).to_dict(orient="records"),
    "review_policy": {"documented_training_records_excluded": len(reviewed_records),
                      "statistical_flags_removed": cfg.REMOVE_FLAGGED_PRICES,
                      "replacement_prices": False},"original_rows": original_rows, "invalid_removed": invalid_count,
    "borong_removed": borong_count, "low_coverage_removed": low_count,
    "excluded_item_codes": [int(c) for c in low_codes],
    "flagged_records_before_optional_removal": flagged_count,
    "remove_flagged_prices": cfg.REMOVE_FLAGGED_PRICES,
    "flagged_removed": flagged_removed,
    "daily_rows_saved": len(model), "item_codes": int(model["item_code"].nunique()),
    "training_end": cfg.TRAIN_END_DATE,
    "flag_rule": {"iqr_multiplier": cfg.IQR_MULTIPLIER,
                  "low_median_multiplier": cfg.LOW_MEDIAN_MULTIPLIER,
                  "high_median_multiplier": cfg.HIGH_MEDIAN_MULTIPLIER},
    "training_price_bounds": stats.reset_index().to_dict(orient="records")}
(cfg.STAGE3_DIR / "cleaning_summary.json").write_text(json.dumps(summary, indent=2))
print("Saved daily rows:", len(model))
print("Next: Notebook 04.")


Training records for source review: 776


,date,state,district,premise_type,premise,premise_code,project_group,item_code,item,unit,item_group,item_category,price,review_reason
0,2025-01-01,W.P. Kuala Lumpur,Bandar Tun Razak,Pasar Raya / Supermarket,GIANT SUPERMARKET ( DESA PETALING ),14832,Eggs,1109,TELUR AYAM GRED A,10 biji,BARANGAN SEGAR,TELUR,0.420000,outside training price bounds; cause unconfirmed
1,2025-01-02,W.P. Kuala Lumpur,Bandar Tun Razak,Pasar Raya / Supermarket,PASARAYA MEMI ZOO ZOO,21071,Eggs,1110,TELUR AYAM GRED B,10 biji,BARANGAN SEGAR,TELUR,0.400000,outside training price bounds; cause unconfirmed
2,2025-01-03,Negeri Sembilan,Seremban,Pasar Raya / Supermarket,FAMILY STORE SIKAMAT,656,Vegetables,1558,SAWI HIJAU,1kg,BARANGAN SEGAR,SAYUR-SAYURAN,1.000000,outside training price bounds; cause unconfirmed
3,2025-01-04,Negeri Sembilan,Seremban,Pasar Raya / Supermarket,FAMILY STORE SIKAMAT,656,Vegetables,1558,SAWI HIJAU,1kg,BARANGAN SEGAR,SAYUR-SAYURAN,1.000000,outside training price bounds; cause unconfirmed
4,2025-01-05,Negeri Sembilan,Seremban,Pasar Raya / Supermarket,FAMILY STORE SIKAMAT,656,Vegetables,1558,SAWI HIJAU,1kg,BARANGAN SEGAR,SAYUR-SAYURAN,1.000000,outside training price bounds; cause unconfirmed
5,2025-01-05,Sarawak,Song,Pasar Basah,PASAR BASAH SONG,16077,Eggs,1109,TELUR AYAM GRED A,10 biji,BARANGAN SEGAR,TELUR,0.466667,outside training price bounds; cause unconfirmed
6,2025-01-06,Negeri Sembilan,Seremban,Pasar Raya / Supermarket,FAMILY STORE SIKAMAT,656,Vegetables,1558,SAWI HIJAU,1kg,BARANGAN SEGAR,SAYUR-SAYURAN,1.000000,outside training price bounds; cause unconfirmed
7,2025-01-06,Pulau Pinang,Timur Laut,Kedai Runcit,MDJ ABDUL NAZAR KEDAI RUNCIT,2130,Vegetables,1558,SAWI HIJAU,1kg,BARANGAN SEGAR,SAYUR-SAYURAN,380.000000,outside training price bounds; cause unconfirmed
8,2025-01-06,Terengganu,Kuala Terengganu,Pasar Raya / Supermarket,SABASUN BANDAR KUALA TERENGGANU,4454,Vegetables,114,TOMATO,1kg,BARANGAN SEGAR,SAYUR-SAYURAN,0.990000,outside training price bounds; cause unconfirmed
9,2025-01-06,Terengganu,Kuala Nerus,Pasar Raya / Supermarket,SABASUN WAKAF TEMBESU,14564,Vegetables,114,TOMATO,1kg,BARANGAN SEGAR,SAYUR-SAYURAN,0.990000,outside training price bounds; cause unconfirmed


Saved daily rows: 4722617
Next: Notebook 04.


## 4. Review training source prices without changing the experiment

**For an already completed run:** select and run just the following cell. It imports its own dependencies and uses the remembered run; do not rerun 01 or the earlier cleaning cells to inspect an existing experiment. During a new preparation, normal Run All includes this section.

This evidence table combines statistical flags with the documented conservative exclusions. It checks the verified integrated snapshot, shows adjacent observations at the same shop/item **within the training period**, and records the original lookup unit and conversion divisor. The implied original price is reconstructed from the integrated price; it is **not an independently downloaded raw record**. Adjacent prices can be suspicious too, so none of these comparisons confirms an error or a replacement price.

The only new file is `03_cleaned/training_source_context.csv`. It is regenerated evidence, not an editable decision log. Record reviewed case IDs, evidence and conclusions in your research notes: **confirmed error**, **plausible observation**, or **unresolved**. A confirmation needs independent source/unit evidence. Keep unresolved prices; any confirmed correction belongs in a separate experiment. No labels, validation/test observations, prepared data or models are changed.

Review priorities are descriptive investigation aids, not learned cleaning rules. Priority 1 means a tenfold-or-larger deviation in either direction from both neighbouring prices, where both observations are within seven days and differ by at most 10% of the previous price. Priority 2 means a tenfold-or-larger deviation from at least one neighbour; priority 3 covers remaining cases. These convenience thresholds do not confirm errors, economic surges or replacement values. Missing neighbours can hide a problematic case. The monthly source URL supports manual checking; it has not been fetched or independently verified by this cell. No new exclusion policy is applied.


In [2]:
# Read-only review of existing training evidence; safe after Notebook 06.
import pandas as pd
import numpy as np
import json
import hashlib
import pricecatcher_config as cfg
from importlib import reload
cfg = reload(cfg)
source_manifest = cfg.verify_source_snapshot()
cleaning = json.loads((cfg.STAGE3_DIR / "cleaning_summary.json").read_text())
source_hash = hashlib.sha256((cfg.STAGE1_DIR / "source_manifest.json").read_bytes()).hexdigest()
if cleaning["preparation"]["source_manifest_sha256"] != source_hash:
    raise ValueError("Cleaning evidence belongs to another source snapshot. Restore its matching files.")
training_end = pd.Timestamp(cleaning["training_end"])
keys = ["date", "premise_code", "item_code"]
cases = pd.read_csv(cfg.STAGE3_DIR / "training_price_review.csv", parse_dates=["date"])
cases["date"] = pd.to_datetime(cases["date"])
cases["price"] = pd.to_numeric(cases["price"], errors="raise")
cases["existing_policy"] = "statistical flag retained or invalid price removed; see review_reason"
excluded = pd.DataFrame(cleaning["reviewed_daily_records"])
if not excluded.empty:
    excluded["date"] = pd.to_datetime(excluded["date"])
    excluded["review_reason"] = "documented suspected tenfold issue; not independently confirmed"
    excluded["existing_policy"] = "documented conservative exclusion"
    cases = pd.concat([cases, excluded], ignore_index=True)
if cases[keys].isna().any().any() or cases.duplicated(keys).any():
    raise ValueError("Review evidence has missing or duplicate case keys.")
if cases["date"].gt(training_end).any():
    raise ValueError("Source review must contain training observations only.")

# Load only affected shop/item histories, with a strict training date filter.
pairs = cases[["premise_code", "item_code"]].drop_duplicates()
history = pd.read_parquet(cfg.STAGE1_COMBINED_PARQUET, columns=keys + ["price", "unit"],
    filters=[("date", "<=", training_end), ("premise_code", "in", pairs["premise_code"].tolist()),
             ("item_code", "in", pairs["item_code"].tolist())])
history["date"] = pd.to_datetime(history["date"])
history = history.merge(pairs, on=["premise_code", "item_code"], validate="many_to_one")
if history.duplicated(keys).any() or history["date"].gt(training_end).any():
    raise ValueError("Source history has duplicate keys or observations beyond training.")
history = history.sort_values(["premise_code", "item_code", "date"])
groups = history.groupby(["premise_code", "item_code"])
for label, step in [("previous", 1), ("following", -1)]:
    history[label + "_date"] = groups["date"].shift(step)
    history[label + "_price"] = groups["price"].shift(step)
history = history.rename(columns={"price": "snapshot_price", "unit": "snapshot_unit"})
evidence = cases.merge(history, on=keys, how="left", validate="one_to_one", indicator=True)
if not evidence["_merge"].eq("both").all() or not np.isclose(
        evidence["price"], evidence["snapshot_price"], rtol=0, atol=1e-12, equal_nan=True).all():
    raise ValueError("Review prices do not match the verified integrated snapshot.")
if not evidence["unit"].eq(evidence["snapshot_unit"]).all():
    raise ValueError("Review units do not match the verified integrated snapshot.")
evidence = evidence.drop(columns=["_merge", "snapshot_price", "snapshot_unit"])
lookup = pd.read_parquet(cfg.LOOKUP_DIR / "lookup_item.parquet", columns=["item_code", "unit"])
lookup = lookup.rename(columns={"unit": "source_lookup_unit"})
evidence = evidence.merge(lookup, on="item_code", how="left", validate="many_to_one")
if evidence["source_lookup_unit"].isna().any():
    raise ValueError("Review items have no source lookup unit.")
expected_egg_unit = evidence["item_code"].map(
    {code: "30 biji" if code in cfg.EGG_30_CODES else "10 biji" for code in cfg.ALL_EGG_CODES})
egg = expected_egg_unit.notna()
if not evidence.loc[egg, "source_lookup_unit"].str.strip().str.lower().eq(expected_egg_unit[egg]).all():
    raise ValueError("Egg lookup unit disagrees with the documented conversion; investigate first.")
evidence["conversion_divisor"] = np.where(evidence["item_code"].isin(cfg.EGG_30_CODES), 3, 1)
evidence["implied_original_price_not_raw_verified"] = evidence["price"] * evidence["conversion_divisor"]
evidence["days_since_previous"] = (evidence["date"] - evidence["previous_date"]).dt.days
evidence["days_until_following"] = (evidence["following_date"] - evidence["date"]).dt.days
evidence["case_id"] = (evidence["date"].dt.strftime("%Y-%m-%d") + "_"
    + evidence["premise_code"].astype(str) + "_" + evidence["item_code"].astype(str))
evidence["review_status"] = "unresolved; independent source evidence required"
evidence["training_end"] = training_end.date().isoformat()
evidence["source_snapshot_sha256"] = source_manifest["source_snapshot_sha256"]
# Review priority only: these retrospective comparisons never remove or replace prices.
previous = evidence["previous_price"]
following = evidence["following_price"]
positive_neighbours = previous.gt(0) & following.gt(0)
nearby = evidence["days_since_previous"].between(1, 7) & evidence["days_until_following"].between(1, 7)
stable_neighbours = positive_neighbours & (previous - following).abs().le(0.10 * previous)
evidence["ratio_to_previous"] = evidence["price"] / previous.where(previous.gt(0))
evidence["ratio_to_following"] = evidence["price"] / following.where(following.gt(0))
large_previous = evidence["ratio_to_previous"].ge(10 - 1e-8) | evidence["ratio_to_previous"].le(0.1 + 1e-8)
large_following = evidence["ratio_to_following"].ge(10 - 1e-8) | evidence["ratio_to_following"].le(0.1 + 1e-8)
isolated = nearby & stable_neighbours & large_previous & large_following
large_deviation = large_previous | large_following
evidence["review_priority"] = np.select([isolated, large_deviation], [1, 2], default=3)
evidence["priority_reason"] = np.select([isolated, large_deviation],
    ["large deviation between nearby similar prices", "large deviation from at least one neighbour"],
    default="other flagged price or insufficient neighbouring evidence")
evidence["source_month_url"] = evidence["date"].dt.strftime("%Y-%m").map(
    lambda month: cfg.BASE_PRICE_URL.format(month=month))
evidence = evidence.sort_values(["review_priority", "project_group", "item_code", "premise_code", "date"])
review_path = cfg.STAGE3_DIR / "training_source_context.csv"
evidence.to_csv(review_path, index=False)
print("Run folder:", cfg.PROJECT_ROOT)
print("Training-only cases:", len(evidence), "| Cutoff:", training_end.date())
print("No price corrections, labels or model files were changed.")
display(evidence.groupby(["review_priority", "priority_reason"]).size().rename("cases").to_frame())
display(evidence.groupby(["project_group", "existing_policy"]).size().rename("cases").to_frame())
display(evidence[["review_priority", "case_id", "item", "price", "unit", "source_lookup_unit", "conversion_divisor",
    "implied_original_price_not_raw_verified", "previous_date", "previous_price", "following_date",
    "following_price", "review_status"]].head(20))
print("Full evidence table:", review_path)


Run folder: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240
Training-only cases: 782 | Cutoff: 2025-12-31
No price corrections, labels or model files were changed.


,,cases
review_priority,priority_reason,
1,large deviation between nearby similar prices,60
2,large deviation from at least one neighbour,27
3,other flagged price or insufficient neighbouring evidence,695


cases
project_group existing_policy                                          
Chicken       statistical flag retained or invalid price remo...     12
Cooking Oil   statistical flag retained or invalid price remo...     14
Eggs          statistical flag retained or invalid price remo...     32
Flour         statistical flag retained or invalid price remo...      3
Rice          statistical flag retained or invalid price remo...      4
Sugar         documented conservative exclusion                       2
              statistical flag retained or invalid price remo...      2
Vegetables    documented conservative exclusion                       4
              statistical flag retained or invalid price remo...    709

,review_priority,case_id,item,price,unit,source_lookup_unit,conversion_divisor,implied_original_price_not_raw_verified,previous_date,previous_price,following_date,following_price,review_status
401,1,2025-06-26_424_1,AYAM BERSIH - STANDARD,880.000000,1kg,1kg,1,880.00,2025-06-25,8.800000,2025-06-28,8.800000,unresolved; independent source evidence required
657,1,2025-10-05_766_1,AYAM BERSIH - STANDARD,80.020000,1kg,1kg,1,80.02,2025-10-04,8.000000,2025-10-06,8.000000,unresolved; independent source evidence required
706,1,2025-10-22_21823_1,AYAM BERSIH - STANDARD,0.860000,1kg,1kg,1,0.86,2025-10-21,8.690000,2025-10-23,8.690000,unresolved; independent source evidence required
273,1,2025-03-04_21196_2,AYAM BERSIH - SUPER,1.120000,1kg,1kg,1,1.12,2025-03-03,11.200000,2025-03-05,11.200000,unresolved; independent source evidence required
750,1,2025-12-03_21196_2,AYAM BERSIH - SUPER,1.100000,1kg,1kg,1,1.10,2025-12-02,11.500000,2025-12-04,11.500000,unresolved; independent source evidence required
52,1,2025-01-26_2007_118,TELUR AYAM GRED A (BERAT 65.0 GM HINGGA 69.9 G...,420.000000,10 biji,10 biji,1,420.00,2025-01-25,4.200000,2025-01-27,4.200000,unresolved; independent source evidence required
5,1,2025-01-05_16077_1109,TELUR AYAM GRED A,0.466667,10 biji,30 biji,3,1.40,2025-01-04,4.700000,2025-01-06,4.700000,unresolved; independent source evidence required
692,1,2025-10-13_21223_1109,TELUR AYAM GRED A,500.000000,10 biji,30 biji,3,1500.00,2025-10-06,5.000000,2025-10-16,4.766667,unresolved; independent source evidence required
48,1,2025-01-20_1714_1110,TELUR AYAM GRED B,0.400000,10 biji,30 biji,3,1.20,2025-01-19,4.000000,2025-01-22,4.000000,unresolved; independent source evidence required
359,1,2025-05-26_20689_1110,TELUR AYAM GRED B,0.400000,10 biji,30 biji,3,1.20,2025-05-19,4.000000,2025-05-29,4.000000,unresolved; independent source evidence required


Full evidence table: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/experiments/holdout/03_cleaned/training_source_context.csv
